# 🏛️ Masterclass 05: Advanced Retrieval, Hybrid Search, & Re-ranking
### *Google Colab Ready | AI Engineer Interview Preparation (3–5 YOE)*

---

## 📌 Executive Summary
In production RAG systems, **naive dense vector retrieval fails** when faced with exact alphanumeric codes, rare vocabulary, or keyword constraints.
Relying solely on vector similarity yields unpredictable hallucinations and irrelevant chunk injection.

This master notebook centralizes the complete engineering theory, mathematical formulas, and verified Python implementations for:
1. **The Duality of Retrieval**: Dense Vector Search (Bi-Encoders) vs Sparse Keyword Search (BM25).
2. **The BM25 Probabilistic Ranking Algorithm**: Term frequency saturation ($k_1$) and document length normalization ($b$).
3. **The Score Calibration Problem & Reciprocal Rank Fusion (RRF)**: Merging rank positions mathematically without arbitrary score normalization.
4. **Bi-Encoders vs Cross-Encoders**: Why dual-encoder dot products are fast for candidate generation, while cross-attention re-rankers provide precision.
5. **Query Transformations**: Query Expansion and HyDE (Hypothetical Document Embeddings).
6. **The Multi-Stage Production Retrieval Pipeline**: Ingestion $\to$ Dual Candidate Retrieval $\to$ RRF $\to$ Cross-Encoder $\to$ Top-$K$ LLM Injection.
7. **Senior AI Interview Q&A & Architecture Decisions**


---
## ⚙️ 1. Environment Setup (Google Colab & Local)
Run the cell below to bootstrap dependencies (`numpy`) and initialize the test environment.


In [ ]:
import sys
import os
import math
import re
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple
import numpy as np

# Optional packages for real Cohere / HuggingFace Cross-Encoders:
# !pip install numpy sentence-transformers cohere rank-bm25

print('NumPy version:', np.__version__)
print('Advanced Retrieval environment initialized successfully.')


---
## 📚 2. Setting Up an Enterprise Test Document Corpus
To test hybrid retrieval, we create a realistic enterprise technical document corpus featuring:
- Exact technical SKU / error codes (`ERR_CONNECTION_RESET`, `K8S-POD-OOM-137`).
- Conceptual and semantic explanations (paraphrased descriptions without exact keywords).


In [ ]:
corpus = [
    {
        'id': 'doc_1',
        'title': 'Kubernetes Pod Exit Code 137',
        'content': 'Exit code K8S-POD-OOM-137 indicates that a container was terminated by the Linux kernel OOM-killer due to exceeding memory resource limits specified in deployment manifests.'
    },
    {
        'id': 'doc_2',
        'title': 'Network Socket Connection Reset Error',
        'content': 'The error ERR_CONNECTION_RESET occurs when the remote TCP peer forcibly closes an active network socket connection due to keep-alive timeouts or intermediate firewall drops.'
    },
    {
        'id': 'doc_3',
        'title': 'High Memory Usage in Docker Containers',
        'content': 'When microservices leak heap memory, background workers consume excessive RAM until the operating system shuts down the process to prevent system-wide instability.'
    },
    {
        'id': 'doc_4',
        'title': 'Network Latency and Firewall Configuration',
        'content': 'Network packets dropped by security groups or cloud VPC access control lists manifest as dropped TCP handshakes and intermittent connection resets.'
    },
    {
        'id': 'doc_5',
        'title': 'Employee Wellness & Remote Work Stipends',
        'content': 'Full-time staff members are eligible for an annual home office allowance and mental health counseling benefits through our third-party partner.'
    }
]

print(f'Loaded enterprise corpus containing {len(corpus)} documents.')


---
## 🔍 3. Sparse Keyword Retrieval: The BM25 Algorithm

### Mathematical Formulation:
BM25 (Best Matching 25) calculates the relevance score of document $D$ for query $Q = (q_1, q_2, \dots, q_n)$:

$$\text{Score}_{BM25}(D, Q) = \sum_{i=1}^n \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

Where:
- **$\text{IDF}(q_i) = \ln\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$**: Inverse Document Frequency. Rare terms receive high weight; ubiquitous terms receive low weight.
- **$f(q_i, D)$**: Term Frequency of word $q_i$ in document $D$.
- **$k_1$ (typically $1.2 - 2.0$)**: Term frequency saturation parameter. Prevents a document that repeats a word 50 times from gaining an unfair advantage over one that mentions it 3 times.
- **$b$ (typically $0.75$)**: Length normalization penalty. Penalizes long documents that mention keywords purely by virtue of being lengthy.


In [ ]:
class PurePythonBM25:
    def __init__(self, documents: List[Dict[str, str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.documents = documents
        self.N = len(documents)
        self.doc_lengths = []
        self.doc_token_counts = []
        self.doc_freqs = defaultdict(int)
        self._index()

    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\w+', text.lower())

    def _index(self):
        total_tokens = 0
        for doc in self.documents:
            tokens = self._tokenize(doc['content'])
            self.doc_lengths.append(len(tokens))
            total_tokens += len(tokens)
            counts = Counter(tokens)
            self.doc_token_counts.append(counts)
            for token in counts.keys():
                self.doc_freqs[token] += 1
        self.avgdl = total_tokens / self.N if self.N > 0 else 1.0

    def _idf(self, term: str) -> float:
        df = self.doc_freqs.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)

    def search(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        query_tokens = self._tokenize(query)
        scores = []
        for idx, doc in enumerate(self.documents):
            score = 0.0
            doc_len = self.doc_lengths[idx]
            counts = self.doc_token_counts[idx]
            for token in query_tokens:
                if token in counts:
                    tf = counts[token]
                    idf = self._idf(token)
                    numerator = tf * (self.k1 + 1)
                    denominator = tf + self.k1 * (1 - self.b + self.b * (doc_len / self.avgdl))
                    score += idf * (numerator / denominator)
            scores.append((doc['id'], round(score, 4)))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

# Instantiate and test BM25 on exact keyword query
bm25 = PurePythonBM25(corpus)
exact_query = 'What does ERR_CONNECTION_RESET mean?'
bm25_matches = bm25.search(exact_query, top_k=3)
print('=== BM25 KEYWORD SEARCH RESULTS ===')
print('Query:', exact_query)
for rank, (doc_id, score) in enumerate(bm25_matches, 1):
    print(f'Rank {rank}: {doc_id} (BM25 Score: {score})')


---
## 🧠 4. Dense Semantic Vector Retrieval (Bi-Encoder Simulation)

### The Dense Mechanism:
Dense embeddings project queries and chunks into a continuous hypersphere.
While BM25 requires exact word overlap, dense vectors excel at **synonyms, concepts, and semantic paraphrases**:


In [ ]:
def mock_dense_embed(text: str) -> np.ndarray:
    """
    Simulated 16-dimensional dense embedding model.
    Dims 0-7: Infrastructure & Out-Of-Memory semantics.
    Dims 8-15: Networking & TCP socket semantics.
    """
    vec = np.zeros(16)
    t = text.lower()
    if any(w in t for w in ['memory', 'ram', 'oom', 'leak', 'killed', 'container', 'instability']):
        vec[0:8] += [0.85, 0.92, 0.78, 0.88, 0.90, 0.82, 0.79, 0.86]
    if any(w in t for w in ['tcp', 'network', 'socket', 'connection', 'reset', 'firewall', 'packets']):
        vec[8:16] += [0.88, 0.95, 0.76, 0.89, 0.91, 0.84, 0.77, 0.83]
    if any(w in t for w in ['wellness', 'health', 'remote', 'stipend']):
        vec[0:4] += [0.1, 0.2, 0.1, 0.2]
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else np.full(16, 0.25)

class DenseVectorSearchEngine:
    def __init__(self, documents: List[Dict[str, str]]):
        self.documents = documents
        self.embeddings = [mock_dense_embed(d['content']) for d in documents]

    def search(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        q_vec = mock_dense_embed(query)
        scores = []
        for idx, doc in enumerate(self.documents):
            d_vec = self.embeddings[idx]
            cos_sim = float(np.dot(q_vec, d_vec))
            scores.append((doc['id'], round(cos_sim, 4)))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]

dense_engine = DenseVectorSearchEngine(corpus)
semantic_query = 'Why did the operating system terminate my application due to high RAM?'
dense_matches = dense_engine.search(semantic_query, top_k=3)
print('=== DENSE VECTOR SEARCH RESULTS ===')
print('Query:', semantic_query)
for rank, (doc_id, score) in enumerate(dense_matches, 1):
    print(f'Rank {rank}: {doc_id} (Cosine Similarity: {score})')


---
## ⚖️ 5. Merging Results: Reciprocal Rank Fusion (RRF)

### The Normalization Dilemma:
- Dense vector scores are bounded in $[0, 1]$.
- BM25 scores are unbounded $[0, \infty)$ and vary wildly depending on query length and term rarity.
- Combining them using linear interpolation ($\alpha \cdot \text{Score}_v + (1 - \alpha) \cdot \text{Score}_b$) produces unstable results across dynamic queries.

### The RRF Solution:
Reciprocal Rank Fusion operates **strictly on rank positions**, ignoring raw scores:

$$\text{RRF Score}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$

Where:
- $r_m(d)$: The 1-based rank position of document $d$ in retrieval system $m$ ($1, 2, 3, \dots$).
- $k = 60$: The smoothing constant that prevents top-rank outliers from overpowering consensus across multiple systems.


In [ ]:
def reciprocal_rank_fusion(ranked_lists: List[List[Tuple[str, float]]], k: int = 60) -> List[Tuple[str, float]]:
    """
    Combines multiple ranked lists using the Reciprocal Rank Fusion formula.
    ranked_lists: List of [(doc_id, raw_score), ...]
    """
    rrf_scores = defaultdict(float)
    
    for system_ranking in ranked_lists:
        for rank_1_indexed, (doc_id, _) in enumerate(system_ranking, 1):
            # RRF formula: 1 / (k + rank)
            rrf_scores[doc_id] += 1.0 / (k + rank_1_indexed)
            
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_id, round(score, 6)) for doc_id, score in sorted_results]

# Run hybrid search for a query requiring both exact code and semantic understanding
hybrid_query = 'Diagnose K8S-POD-OOM-137 memory limits and RAM crashes'
bm25_res = bm25.search(hybrid_query, top_k=5)
dense_res = dense_engine.search(hybrid_query, top_k=5)

fused_results = reciprocal_rank_fusion([bm25_res, dense_res], k=60)

print('=== RECIPROCAL RANK FUSION (RRF) COMBINED RESULTS ===')
print('BM25 Top Matches: ', [d for d, _ in bm25_res[:3]])
print('Dense Top Matches:', [d for d, _ in dense_res[:3]])
print('\nFinal Consolidated RRF Ranking:')
for rank, (doc_id, rrf_score) in enumerate(fused_results[:3], 1):
    print(f'Rank {rank}: {doc_id} (RRF Score: {rrf_score})')


---
## 🎯 6. Cross-Encoder Re-ranking: Precision Filtering

### Bi-Encoder vs Cross-Encoder Architecture:
- **Bi-Encoder**: Encodes Query and Document **independently**. Fast ($\mathcal{O}(1)$ vector lookup), but missing token-to-token cross-attention.
- **Cross-Encoder**: Feeds Query and Document **together** into a single Transformer:
  $$\text{Input} = [\text{CLS}] \circ \text{Query} \circ [\text{SEP}] \circ \text{Document} \circ [\text{EOS}]$$
  Every token in the query attends directly to every token in the document, capturing nuanced semantic qualifiers and negation.


In [ ]:
class MockCrossEncoderReranker:
    """
    Simulates a Cross-Encoder (e.g. Cohere Rerank or BGE-Reranker-Large)
    evaluating deep cross-attention interaction between query and candidate text.
    """
    def rerank(self, query: str, candidate_docs: List[Dict[str, str]], top_k: int = 2) -> List[Tuple[Dict[str, str], float]]:
        scored = []
        q_tokens = set(re.findall(r'\w+', query.lower()))
        for doc in candidate_docs:
            d_tokens = set(re.findall(r'\w+', doc['content'].lower()))
            overlap = len(q_tokens.intersection(d_tokens))
            # Simulate deep attention: exact token co-occurrence + length penalty
            cross_attention_score = (overlap * 0.25) + (0.5 if '137' in doc['content'] and '137' in query else 0.0)
            scored.append((doc, round(min(0.99, cross_attention_score), 4)))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

# Rerank top candidate documents from RRF pool
candidate_docs = [d for d in corpus if d['id'] in [f[0] for f in fused_results[:3]]]
reranker = MockCrossEncoderReranker()
final_reranked = reranker.rerank(hybrid_query, candidate_docs, top_k=2)

print('=== CROSS-ENCODER SECOND-STAGE RE-RANKING ===')
print('Query:', hybrid_query)
for rank, (doc, score) in enumerate(final_reranked, 1):
    print(f'Rank {rank} (Re-rank Score: {score}): [{doc["id"]}] {doc["title"]}')
    print(f'   Snippet: {doc["content"][:90]}...\n')


---
## 💡 7. Query Transformation: HyDE (Hypothetical Document Embeddings)

### How HyDE Works:
When queries are short (e.g., *"K8S crash"*), dense vectors perform poorly against detailed multi-sentence paragraphs.
- **Step 1**: Prompt an LLM to generate a hypothetical answer passage.
- **Step 2**: Embed the **hypothetical passage** instead of the user query.
- **Step 3**: The hypothetical vector and real document vectors share the same grammatical density and vocabulary distribution!


In [ ]:
def simulate_hyde_pipeline(short_user_query: str) -> str:
    """
    Simulates LLM-based Hypothetical Document Generation (Gao et al., 2022).
    """
    print(f'User Query: "{short_user_query}"')
    print('Generating Hypothetical Document via LLM...')
    # Simulated LLM hallucinated response that structurally matches technical manuals
    hypothetical_doc = (
        'In cloud container orchestration, when a pod experiences memory exhaustion, '
        'the cgroup limits trigger an out-of-memory SIGKILL signal resulting in container termination.'
    )
    print('Hypothetical Passage Generated:', hypothetical_doc)
    return hypothetical_doc

hyde_passage = simulate_hyde_pipeline('pod crash memory')
hyde_matches = dense_engine.search(hyde_passage, top_k=2)
print('\n=== RETRIEVAL USING HYPOTHETICAL PASSAGE EMBEDDING ===')
for rank, (doc_id, score) in enumerate(hyde_matches, 1):
    print(f'Rank {rank}: {doc_id} (Cosine Similarity: {score})')


---
## 🏗️ 8. Putting It All Together: The Multi-Stage Production Retrieval Pipeline

```
[ User Query ]
       │
       ├───────────────────────────────────────────┐
       ▼                                           ▼
[ Dense Vector Search ]                     [ Sparse BM25 Search ]
 (pgvector: Top 25 Chunks)                   (Elasticsearch: Top 25 Chunks)
       │                                           │
       └─────────────────────┬─────────────────────┘
                             │
                             ▼
          [ Reciprocal Rank Fusion (RRF, k=60) ]
                             │
                             ▼ (Top 15 Candidate Pool)
             [ Cross-Encoder Re-ranker (Cohere / BGE) ]
                             │
                             ▼
             [ Top 3 Ultra-Precise Final Chunks ]
                             │
                             ▼
                [ Injected into LLM Prompt ]
```


In [ ]:
class ProductionMultiStageRetriever:
    def __init__(self, corpus: List[Dict[str, str]], k_rrf: int = 60):
        self.corpus = {d['id']: d for d in corpus}
        self.bm25 = PurePythonBM25(corpus)
        self.dense = DenseVectorSearchEngine(corpus)
        self.reranker = MockCrossEncoderReranker()
        self.k_rrf = k_rrf

    def retrieve(self, query: str, top_k_final: int = 2) -> List[Dict[str, Any]]:
        # Stage 1: Parallel Dense + Sparse Retrieval (Fetch top 4 each)
        bm25_candidates = self.bm25.search(query, top_k=4)
        dense_candidates = self.dense.search(query, top_k=4)

        # Stage 2: Reciprocal Rank Fusion
        fused = reciprocal_rank_fusion([bm25_candidates, dense_candidates], k=self.k_rrf)
        candidate_pool = [self.corpus[doc_id] for doc_id, _ in fused[:4]]

        # Stage 3: Deep Cross-Encoder Re-ranking
        reranked = self.reranker.rerank(query, candidate_pool, top_k=top_k_final)
        
        return [{'doc': doc, 'final_score': score} for doc, score in reranked]

pipeline = ProductionMultiStageRetriever(corpus)
final_chunks = pipeline.retrieve('How to fix K8S-POD-OOM-137 memory limits in Kubernetes?', top_k_final=2)

print('=== END-TO-END PRODUCTION RETRIEVAL EXECUTION ===')
for idx, item in enumerate(final_chunks, 1):
    print(f'Final Injected Chunk {idx} (Score {item["final_score"]}):')
    print(f'Title: {item["doc"]["title"]}')
    print(f'Text:  {item["doc"]["content"]}\n')


---
## 🎯 9. Senior AI Interview Questions: Retrieval & Re-ranking

### Q1: Why can't we simply normalize BM25 scores between $[0, 1]$ and compute a weighted sum with vector similarity?
> **Answer**: BM25 scores are unbounded $[0, \infty)$ and their dynamic range fluctuates drastically depending on the rarity of terms in each specific query. Min-Max normalization fails because the maximum possible BM25 score for an arbitrary query cannot be determined a priori. If one query scores between $[0, 4]$ and another between $[0, 40]$, linear weights ($\alpha$) will unpredictably favor vector search in one query and BM25 in another. **Reciprocal Rank Fusion (RRF)** solves this by relying purely on rank positions ($1 / (k + r)$), which are scale-invariant.

---

### Q2: What is the architectural difference between a Bi-Encoder and a Cross-Encoder?
> **Answer**:
> - **Bi-Encoder** encodes the Query and Document separately into two isolated vectors without cross-token interaction. Similarity is a dot product. It is computationally lightweight (milliseconds across millions of vectors), but misses fine-grained token relationships.
> - **Cross-Encoder** concatenates Query and Document into a single sequence ($[\text{CLS}] Q [\text{SEP}] D [\text{EOS}]$) and passes it through full Transformer self-attention. Every token in the query attends directly to every token in the document. It is computationally expensive ($\mathcal{O}(N)$ passes through the neural network), making it ideal as a **second-stage re-ranker on the top 20-30 candidates**.

---

### Q3: When does HyDE (Hypothetical Document Embeddings) fail in production?
> **Answer**: HyDE fails when the user query relates to **brand-new, proprietary, or rapidly changing domain knowledge** that the LLM has never seen. In these cases, the LLM hallucinates incorrect terminology, steering the embedding vector toward completely wrong clusters in vector space. HyDE is best suited for conceptual, open-domain questions where grammatical structure is more important than exact entity accuracy.
